# Automatic Classification

## Business Goals
   Develop an automatic classification model that predicts the automation 
   risk level (High/Medium/Low) of different occupations by 2030, based on 
   characteristics such as salary, experience, education, AI exposure, and 
   required skills.

## Selected Algorithms:

### K Nearest Neighbors (KNN)
- Instance-based learning, no training phase
- Captures non-linear decision boundaries
- Simple and intuitive approach: classifies based on similar occupations
- Good for understanding local patterns in the data
- Note: Requires feature scaling due to different ranges (salary vs skills)

### Logistic Regression
- Simple, fast, and highly interpretable model
- Establishes a baseline for linear relationships
- Provides probability outputs and feature coefficients
- Computationally efficient for training and prediction
- Easy to explain to stakeholders

## Necessary Imports

In [112]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay
)

# Configuration
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

## Load data

In [113]:
df = pd.read_csv('AI_Impact_on_Jobs_2030.csv')

## Data cleaning

### Missing values

#### Check:

How much missing values per column exist in the dataset.

In [114]:
df_clean = df.copy()
print("\nMissing values per column:")
missing_counts = df_clean.isnull().sum()
print(missing_counts)


Missing values per column:
Job_Title                      0
Average_Salary                 0
Years_Experience               0
Education_Level                0
AI_Exposure_Index              0
Tech_Growth_Factor             0
Automation_Probability_2030    0
Risk_Category                  0
Skill_1                        0
Skill_2                        0
Skill_3                        0
Skill_4                        0
Skill_5                        0
Skill_6                        0
Skill_7                        0
Skill_8                        0
Skill_9                        0
Skill_10                       0
dtype: int64


#### Remove:

Removing missing values from the dataset.

In [115]:
df_clean = df_clean.dropna()

### Removing duplicates

#### Check:
How much duplicates are in the dataset.

In [116]:
duplicates_count = df_clean.duplicated().sum()
print(f"\nDuplicate rows found: {duplicates_count}")


Duplicate rows found: 0


#### Remove
Removing duplicates values from dataset

In [117]:
duplicates_count = df_clean.duplicated().sum()

if duplicates_count > 0:
    df_clean = df_clean.drop_duplicates()
    print("Duplicates have been removed.")

print(f"New total rows: {len(df_clean)}")

New total rows: 3000


## Label Enconding

Converts categorical education titles into numerical ranks (1−4). This preserves the natural hierarchy of the data (e.g., a PhD is 'higher' than a Bachelor's)

In [118]:
print("\nUnique Education Levels:")
print(df['Education_Level'].unique())

# Define ordinal mapping
education_mapping = {
    'High School': 1,
    "Bachelor's": 2,
    "Master's": 3,
    'PhD': 4
}

df['Education_Level_Encoded'] = df['Education_Level'].map(education_mapping)

print("\nEducation Level Mapping:")
for key, value in education_mapping.items():
    print(f"  {key}: {value}")

# Drop original Education_Level
df = df.drop('Education_Level', axis=1)

print(f"\nEducation_Level encoded and original column removed")
print(f"Current columns: {list(df.columns)}")


Unique Education Levels:
["Master's" 'PhD' 'High School' "Bachelor's"]

Education Level Mapping:
  High School: 1
  Bachelor's: 2
  Master's: 3
  PhD: 4

Education_Level encoded and original column removed
Current columns: ['Job_Title', 'Average_Salary', 'Years_Experience', 'AI_Exposure_Index', 'Tech_Growth_Factor', 'Automation_Probability_2030', 'Risk_Category', 'Skill_1', 'Skill_2', 'Skill_3', 'Skill_4', 'Skill_5', 'Skill_6', 'Skill_7', 'Skill_8', 'Skill_9', 'Skill_10', 'Education_Level_Encoded']


### Target / Feature Separation
Separate the dataset into X (the input features) and y (the target result).

In [119]:
X = df.drop(['Risk_Category', 'Job_Title'], axis=1, errors='ignore')
y = df['Risk_Category']

### Encoding target variable

In [120]:
# Check unique values in target
print("\nUnique Risk Categories:")
print(y.unique())
print("\nDistribution:")
print(y.value_counts())

# Map to numbers: Low=0, Medium=1, High=2
risk_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
y_encoded = y.map(risk_mapping)


Unique Risk Categories:
['High' 'Low' 'Medium']

Distribution:
Risk_Category
Medium    1521
High       740
Low        739
Name: count, dtype: int64


### Split Dataset

Split the data into 80% for training and 20% for testing. This is a standard way to give the model plenty of data to learn from while saving a smaller piece to test. By testing the model on data it hasn't seen before, we can make sure it actually understands the patterns and isn't just memorizing the answers.

In [121]:
# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y_encoded, 
    test_size=0.2,        # 20% for testing
    random_state=42,      # for reproducibility
    stratify=y_encoded    # maintain class distribution
)

### Feature Scaling
Put all features on the same scale so the model treats them fairly. 
Without this, features with big numbers (like Salary) would drown out 
features with small numbers (like Age). We use Feature Scaling to 
center the data around 0, making sure the AI looks at the patterns 
rather than just the size of the numbers.

In [122]:
# Initialize the scaler
scaler = StandardScaler()

# Fit on TRAINING data and transform both sets
X_train_scaled = scaler.fit_transform(X_train)  # Learn mean/std from training
X_test_scaled = scaler.transform(X_test)        # Apply same transformation to test

# Convert back to DataFrame for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

# Show example of scaling effect
print("\n")
print("BEFORE Scaling (first training sample):")
print("\n")
print(X_train.iloc[0].round(2))

print("\n")
print("AFTER Scaling (same sample):")
print("\n")
print(X_train_scaled.iloc[0].round(2))



BEFORE Scaling (first training sample):


Average_Salary                 55463.00
Years_Experience                   8.00
AI_Exposure_Index                  0.50
Tech_Growth_Factor                 1.20
Automation_Probability_2030        0.33
Skill_1                            0.94
Skill_2                            0.46
Skill_3                            0.94
Skill_4                            0.11
Skill_5                            0.95
Skill_6                            0.11
Skill_7                            0.46
Skill_8                            0.94
Skill_9                            0.23
Skill_10                           0.03
Education_Level_Encoded            1.00
Name: 1116, dtype: float64


AFTER Scaling (same sample):


Average_Salary                -1.00
Years_Experience              -0.78
AI_Exposure_Index             -0.00
Tech_Growth_Factor             0.69
Automation_Probability_2030   -0.70
Skill_1                        1.51
Skill_2                       -0.11
Skil

### K Neareast Neighbors (KNN)

KNN is a simple but powerful "neighborhood-based" algorithm. Unlike other models that try to learn complex mathematical formulas, KNN works on a very intuitive principle: similarity. It assumes that data points that are close to each other in the "feature space" will likely belong to the same category.

In [123]:
print("MODEL 1: K-NEAREST NEIGHBORS (Non-Linear Baseline)")
print("-"*80)

# Train the model (using k=5 as default)
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_knn = knn_model.predict(X_test_scaled)

# Calculate metrics
knn_accuracy = accuracy_score(y_test, y_pred_knn)
knn_precision = precision_score(y_test, y_pred_knn, average='weighted')
knn_recall = recall_score(y_test, y_pred_knn, average='weighted')
knn_f1 = f1_score(y_test, y_pred_knn, average='weighted')

# Store results
models['KNN'] = knn_model
results['KNN'] = {
    'Accuracy': knn_accuracy,
    'Precision': knn_precision,
    'Recall': knn_recall,
    'F1-Score': knn_f1
}

print(f"\nAccuracy:  {knn_accuracy:.4f}")
print(f"Precision: {knn_precision:.4f}")
print(f"Recall:    {knn_recall:.4f}")
print(f"F1-Score:  {knn_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn, target_names=['Low', 'Medium', 'High']))

MODEL 1: K-NEAREST NEIGHBORS (Non-Linear Baseline)
--------------------------------------------------------------------------------

Accuracy:  0.7267
Precision: 0.7386
Recall:    0.7267
F1-Score:  0.7234

Classification Report:
              precision    recall  f1-score   support

         Low       0.74      0.64      0.69       148
      Medium       0.69      0.83      0.76       304
        High       0.84      0.59      0.70       148

    accuracy                           0.73       600
   macro avg       0.76      0.69      0.71       600
weighted avg       0.74      0.73      0.72       600



### Adjust Hyperparameters

To improve the model's performance, we adjusted the n_neighbors to 8 and set the weights to 'distance'. By choosing 8 neighbors, we ask the model for a "group opinion" rather than relying on just one point, which provides a more stable and balanced view that isn't easily fooled by outliers or "weird" data points. Switching the weights to 'distance' tells the AI that not all neighbors are equal, instead, the closest neighbors get a much "louder" vote than those further away. This ensures that the most similar examples have the most influence on the final result, preventing distant and less-relevant neighbors from ruining the prediction.

In [124]:
print("K-NEAREST NEIGHBORS (Adjusted)")
print("-"*80)

# We are adjusting the "Hyperparameters" here:
# weights='distance': Gives closer neighbors more "voting power"
knn_model_adj = KNeighborsClassifier(n_neighbors=8, weights='distance')
knn_model_adj.fit(X_train_scaled, y_train)

# Make predictions
y_pred_knn_adj = knn_model_adj.predict(X_test_scaled)

# Calculate metrics
knn_adj_accuracy = accuracy_score(y_test, y_pred_knn_adj)
knn_adj_precision = precision_score(y_test, y_pred_knn_adj, average='weighted')
knn_adj_recall = recall_score(y_test, y_pred_knn_adj, average='weighted')
knn_adj_f1 = f1_score(y_test, y_pred_knn_adj, average='weighted')

# Store results
models['KNN Adjusted'] = knn_model_adj
results['KNN Adjusted'] = {
    'Accuracy': knn_adj_accuracy,
    'Precision': knn_adj_precision,
    'Recall': knn_adj_recall,
    'F1-Score': knn_adj_f1
}

print(f"\nAdjusted Accuracy:  {knn_adj_accuracy:.4f}")
print(f"Adjusted Precision: {knn_adj_precision:.4f}")
print(f"Adjusted Recall:    {knn_adj_recall:.4f}")
print(f"Adjusted F1-Score:  {knn_adj_f1:.4f}")

print("\nClassification Report (Adjusted):")
print(classification_report(y_test, y_pred_knn_adj, target_names=['Low', 'Medium', 'High']))

K-NEAREST NEIGHBORS (Adjusted)
--------------------------------------------------------------------------------

Adjusted Accuracy:  0.7617
Adjusted Precision: 0.7740
Adjusted Recall:    0.7617
Adjusted F1-Score:  0.7585

Classification Report (Adjusted):
              precision    recall  f1-score   support

         Low       0.80      0.63      0.70       148
      Medium       0.72      0.87      0.79       304
        High       0.86      0.67      0.75       148

    accuracy                           0.76       600
   macro avg       0.79      0.72      0.75       600
weighted avg       0.77      0.76      0.76       600



### Logistic Regression (Linear Baseline)

Logistic Regression is used as our baseline model to establish a starting point for performance. Even though it has "regression" in the name, it is actually a classification algorithm that works by drawing straight lines (linear boundaries) to separate different categories. It is a great first step because it is fast, easy to understand, and helps us see if the relationship between our features and risk levels is simple or requires a more complex model like KNN.

In [125]:
# Initialize dictionaries to store models and results
models = {}
results = {}

# Train the model
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_lr = lr_model.predict(X_test_scaled)

# Calculate metrics
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr, average='weighted')
lr_recall = recall_score(y_test, y_pred_lr, average='weighted')
lr_f1 = f1_score(y_test, y_pred_lr, average='weighted')

# Store results
models['Logistic Regression'] = lr_model
results['Logistic Regression'] = {
    'Accuracy': lr_accuracy,
    'Precision': lr_precision,
    'Recall': lr_recall,
    'F1-Score': lr_f1
}

print(f"\nAccuracy:  {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall:    {lr_recall:.4f}")
print(f"F1-Score:  {lr_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Low', 'Medium', 'High']))



Accuracy:  0.9900
Precision: 0.9900
Recall:    0.9900
F1-Score:  0.9900

Classification Report:
              precision    recall  f1-score   support

         Low       1.00      0.99      0.99       148
      Medium       0.99      0.99      0.99       304
        High       0.99      0.99      0.99       148

    accuracy                           0.99       600
   macro avg       0.99      0.99      0.99       600
weighted avg       0.99      0.99      0.99       600



### Conclusion: Model Comparison and Final Analysis

After training and testing our models, we see a clear difference in how they handle the data. The Logistic Regression model performed exceptionally well, achieving an nearly perfect 99% accuracy, which suggests that the relationship between the features and the Risk Categories is highly linear and well-defined. While the K-Nearest Neighbors (KNN) model also performed respectably, it struggled more than the linear baseline. However, by adjusting the hyperparameters to 8 neighbors and using distance-based weighting, we successfully pushed the KNN accuracy from 72.67% to 76.17%. This improvement shows that while the linear model is the strongest choice here, fine-tuning the KNN helped it capture the patterns much more effectively, particularly in identifying the "High" and "Low" risk categories.